# Face Anonymization for Photogrammetry Scans

GDPR-compliant face removal pipeline for Einstar 3D photogrammetry scans.
Removes facial geometry while preserving all anatomical landmarks and optode positions.

**Pipeline:**
1. Load scan and detect nasion (auto via MediaPipe, or manual click)
2. Normalize axes, isolate head, detect landmarks, compute face mask + ear coverage
3. Preview the detected region
4. Interactive refinement: click on missed spots to add circular deletion zones
5. Delete facial vertices and visualize result

In [1]:
import logging

import numpy as np
import pyvista as pv
import trimesh
import cedalion
import cedalion.io
import cedalion.plots
import cedalion.dataclasses
from cedalion.geometry.photogrammetry.anonymization import (
    detect_nasion_auto,
    pick_nasion,
    normalize_axes,
    isolate_head,
    detect_landmarks_from_nasion,
    get_facial_region_mask_from_nasion,
)
from cedalion.vtktutils import trimesh_to_vtk_polydata
from cedalion.dataclasses import VTKSurface
from scipy.spatial import KDTree

# Suppress verbose module and MediaPipe logging
logging.getLogger('cedalion').setLevel(logging.WARNING)
logging.getLogger('absl').setLevel(logging.WARNING)

pv.set_jupyter_backend('server')

# === CONFIG ===
SUBJECT_NUMBER = 19
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'
FORCE_MANUAL = False
PROXIMITY_RADIUS = 85.0    # mm (unused by new pipeline, kept for reference)
CIRCLE_RADIUS = 25.0       # mm -- manual click-to-delete radius

# ---------------------------------------------------------------------------
# Helper functions (ported from 50_scan_overview.ipynb)
# ---------------------------------------------------------------------------

def eye_plane_rotation_matrix(r_eye_3d, l_eye_3d, forward_dir):
    """Build the rotation that sends the eye line to +Z, forward to +Y,
    and X = Y x Z (head-up, right-handed). Rows of the returned matrix are
    the new basis vectors in the old frame: p_new = R @ (p_old - center) + center.
    """
    eye_dir = np.asarray(l_eye_3d - r_eye_3d, dtype=float)
    z_new = eye_dir / np.linalg.norm(eye_dir)

    fref = np.asarray(forward_dir, dtype=float)
    fref = fref / np.linalg.norm(fref)
    y_new = fref - np.dot(fref, z_new) * z_new
    y_new = y_new / np.linalg.norm(y_new)

    x_new = np.cross(y_new, z_new)
    x_new = x_new / np.linalg.norm(x_new)
    return np.vstack([x_new, y_new, z_new])


def heuristic_lpa_rpa(head_verts, eye_x, centroid_y):
    """Approximate LPA/RPA as small uncertainty spheres, not single points.

    Expects head_verts in the eye-plane-normalized frame
    (X=head-up, Y=forward, Z=left).
    """
    X_TOL = 20.0
    X_OFFSET = -15.0
    Y_TOL = 50.0
    Y_FORWARD = 35.0
    RADIUS = 15.0

    x_band = np.abs(head_verts[:, 0] - (eye_x + X_OFFSET)) < X_TOL
    y_band = np.abs(head_verts[:, 1] - centroid_y) < Y_TOL
    mask = x_band & y_band
    candidates = head_verts[mask]
    if len(candidates) < 10:
        return None, None

    lpa_center = candidates[int(np.argmax(candidates[:, 2]))].astype(float).copy()
    rpa_center = candidates[int(np.argmin(candidates[:, 2]))].astype(float).copy()
    lpa_center[1] += Y_FORWARD
    rpa_center[1] += Y_FORWARD

    return (
        {"center": lpa_center, "radius": RADIUS},
        {"center": rpa_center, "radius": RADIUS},
    )


## 1. Load scan + detect nasion

Auto-detect nasion via MediaPipe. If no face is found (or `FORCE_MANUAL=True`),
falls back to an interactive picker where the user clicks the nasion.

In [2]:
path = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
surface = cedalion.io.read_einstar_obj(path)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

# Nasion detection
if not FORCE_MANUAL:
    auto_result = detect_nasion_auto(surface)
else:
    auto_result = None

if auto_result is not None:
    nasion, meta = auto_result
    print(f'Auto nasion: method={meta["method"]}, confidence={meta["confidence"]:.2f}')
else:
    print('Opening manual nasion picker...')
    nasion = pick_nasion(surface)
    if nasion is None:
        raise RuntimeError('No nasion selected — cannot proceed')
    meta = {'method': 'manual', 'confidence': 1.0}
    print(f'Manual nasion at: [{nasion[0]:.1f}, {nasion[1]:.1f}, {nasion[2]:.1f}]')

fwd = meta.get('forward_direction')
face_contour = meta.get('face_contour_3d')
eyes = meta.get('eyes')
print(f'Forward direction: {"available" if fwd is not None else "N/A"}')
print(f'Face contour: {len(face_contour)} pts' if face_contour is not None else 'Face contour: N/A')
print(f'Eyes: {"available" if eyes is not None else "N/A"}')

Loaded: 408,990 vertices, 721,382 faces


W0000 00:00:1776003948.935699  266809 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1776003948.982133  266809 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776003948.994993  266835 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.4-arch1.1), renderer: AMD Radeon Graphics (radeonsi, renoir, ACO, DRM 3.64, 6.19.11-arch1-1)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776003949.004314  266819 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776003949.016300  266814 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Auto nasion: method=mediapipe+unified, confidence=0.90
Forward direction: available
Face contour: 35 pts
Eyes: available


## 2. Eye-plane rotation, head isolation, heuristic LPA/RPA, face mask


In [3]:
# --- Eye-plane rotation (replaces normalize_axes) ---
# Eyes come from the same validated sweep as the nasion, so they are
# guaranteed consistent with it (see nasion_detector._find_nose_tip_mediapipe).
if eyes is None:
    raise RuntimeError(
        "No eyes in nasion metadata -- unified sweep failed. "
        "Fall back to manual pipeline."
    )
r_eye_raw, l_eye_raw = eyes

R_eye = eye_plane_rotation_matrix(r_eye_raw, l_eye_raw, fwd)

# Build a rotated TrimeshSurface so isolate_head can work with it.
rotated_mesh = surface.mesh.copy()
raw_verts = np.asarray(rotated_mesh.vertices, dtype=float)
rot_center = raw_verts.mean(axis=0)
rotated_mesh.vertices = (R_eye @ (raw_verts - rot_center).T).T + rot_center
surface_n = cedalion.dataclasses.TrimeshSurface(
    mesh=rotated_mesh, crs=surface.crs, units=surface.units
)
nasion_n = R_eye @ (np.asarray(nasion) - rot_center) + rot_center
r_eye_n  = R_eye @ (r_eye_raw - rot_center) + rot_center
l_eye_n  = R_eye @ (l_eye_raw - rot_center) + rot_center
print(f"Eye-plane rotation applied. Canonical frame: X=up, Y=forward, Z=left")
print(f"  nasion_n: [{nasion_n[0]:.1f}, {nasion_n[1]:.1f}, {nasion_n[2]:.1f}]")

# --- Isolate head (library call strips disconnected fragments automatically) ---
surface_h, _ = isolate_head(surface_n, nasion_n)
print(f"Head isolation: {surface_n.nvertices:,} -> {surface_h.nvertices:,} vertices")

# --- Heuristic LPA/RPA in the eye-plane canonical frame ---
verts = np.asarray(surface_h.mesh.vertices)
head_centroid = verts.mean(axis=0)
eye_x = 0.5 * (r_eye_n[0] + l_eye_n[0])
lpa_heur, rpa_heur = heuristic_lpa_rpa(verts, eye_x, head_centroid[1])
if lpa_heur is None or rpa_heur is None:
    raise RuntimeError("heuristic_lpa_rpa returned None -- ear slice had < 10 candidates")

lpa_c, lpa_r = lpa_heur["center"], lpa_heur["radius"]
rpa_c, rpa_r = rpa_heur["center"], rpa_heur["radius"]
print(f"  LPA heur: center=[{lpa_c[0]:.1f}, {lpa_c[1]:.1f}, {lpa_c[2]:.1f}]  radius={lpa_r:.0f}")
print(f"  RPA heur: center=[{rpa_c[0]:.1f}, {rpa_c[1]:.1f}, {rpa_c[2]:.1f}]  radius={rpa_r:.0f}")

# --- Custom anonymization mask ---
# Z-band: stay strictly inside the ear spheres (exclusive of radii)
z_band = (verts[:, 2] > rpa_c[2] + rpa_r) & (verts[:, 2] < lpa_c[2] - lpa_r)

# Y-cut: front half only, use forehead (nasion Y) as safety margin for cap
y_cut = min(0.5 * (lpa_c[1] + rpa_c[1]), nasion_n[1])
front_half = verts[:, 1] > y_cut

# --- Per-vertex upper X bound from MediaPipe forehead arc ---
# Rotate the face oval into the canonical frame and keep only the upper arc
# (contour points above the nasion -- these trace the forehead boundary).
# Each mesh vertex then gets an upper X limit interpolated from the nearest
# arc point in Z (lateral), so the deletion top follows the real forehead
# instead of a fixed rectangle height.
RECT_HALF_WIDTH = 15.0        # mm, half-width in Z of preserved nasion strip
FOREHEAD_FALLBACK = 50.0      # mm above nasion if contour unavailable

if face_contour is not None:
    contour_n_all = (R_eye @ (np.asarray(face_contour) - rot_center).T).T + rot_center
    forehead_arc = contour_n_all[contour_n_all[:, 0] > nasion_n[0]]
else:
    forehead_arc = np.empty((0, 3))

if len(forehead_arc) >= 3:
    arc_tree = KDTree(forehead_arc[:, 2:3])
    _, arc_idx = arc_tree.query(verts[:, 2:3], k=1)
    forehead_x_per_vertex = forehead_arc[arc_idx, 0]
    arc_source = f"MediaPipe forehead arc ({len(forehead_arc)} pts)"
else:
    forehead_x_per_vertex = np.full(len(verts), nasion_n[0] + FOREHEAD_FALLBACK)
    arc_source = f"fallback (nasion + {FOREHEAD_FALLBACK:.0f}mm)"

# Region A: everything BELOW the nasion, between ears, front half
face_below = (verts[:, 0] < nasion_n[0]) & z_band & front_half

# Region B: between nasion and the per-vertex forehead arc,
#           EXCLUDING the vertical nasion strip (kept for registration)
in_arc_band = (verts[:, 0] >= nasion_n[0]) & (verts[:, 0] < forehead_x_per_vertex)
nasion_strip = (
    (np.abs(verts[:, 2] - nasion_n[2]) < RECT_HALF_WIDTH)
    & in_arc_band
    & front_half
)
face_sides = in_arc_band & z_band & front_half & ~nasion_strip

# Nothing above the arc is deleted.
extended_mask = face_below | face_sides

print(f"\nCustom mask: {extended_mask.sum():,} vertices "
      f"({100 * extended_mask.sum() / len(extended_mask):.1f}%)")
print(f"  below nasion:            {face_below.sum():,}")
print(f"  sides (nasion-arc):      {face_sides.sum():,}")
print(f"  nasion strip (kept):     {nasion_strip.sum():,}")
print(f"  forehead upper bound:    {arc_source}")
print(f"    arc X range:           [{forehead_x_per_vertex.min():.1f}, "
      f"{forehead_x_per_vertex.max():.1f}]")
print(f"  Z band (excl. ears):     ({rpa_c[2] + rpa_r:.1f}, {lpa_c[2] - lpa_r:.1f})")
print(f"  Y cut (front half):      > {y_cut:.1f}")
print(f"  Nasion strip half-width: {RECT_HALF_WIDTH:.0f}mm in Z")

# Downstream cells expect: verts, extended_mask, protected_positions
protected_positions = np.stack([nasion_n, lpa_c, rpa_c])
protected_labels = ["Nz", "LPA_heur", "RPA_heur"]


Eye-plane rotation applied. Canonical frame: X=up, Y=forward, Z=left
  nasion_n: [83.5, 118.9, 564.3]
Head isolation: 408,990 -> 408,990 vertices
  LPA heur: center=[94.6, 0.6, 662.0]  radius=15
  RPA heur: center=[93.9, 14.1, 464.6]  radius=15

Custom mask: 90,443 vertices (22.1%)
  below nasion:            62,625
  sides (nasion-arc):      27,818
  nasion strip (kept):     2,828
  forehead upper bound:    MediaPipe forehead arc (14 pts)
    arc X range:           [101.2, 125.4]
  Z band (excl. ears):     (479.6, 647.0)
  Y cut (front half):      > 7.4
  Nasion strip half-width: 15mm in Z


## 3. Preview anonymization region

In [4]:
pvplt = pv.Plotter()
cedalion.plots.plot_surface(pvplt, surface_h, opacity=1.0)

# Deletion region (red scatter)
if extended_mask.sum() > 0:
    pvplt.add_mesh(
        pv.PolyData(verts[extended_mask]), color="red", point_size=3,
        opacity=0.4, render_points_as_spheres=True,
        label=f"deletion region ({extended_mask.sum():,})",
    )

# Heuristic LPA/RPA as translucent spheres (NOT to be deleted)
pvplt.add_mesh(pv.Sphere(radius=lpa_r, center=lpa_c), color="orange", opacity=0.35)
pvplt.add_point_labels([lpa_c], ["LPA heur"], font_size=14,
                       text_color="orange", bold=True, shape=None)
pvplt.add_mesh(pv.Sphere(radius=rpa_r, center=rpa_c), color="blue", opacity=0.35)
pvplt.add_point_labels([rpa_c], ["RPA heur"], font_size=14,
                       text_color="blue", bold=True, shape=None)

# Nasion
pvplt.add_mesh(pv.Sphere(radius=3, center=nasion_n), color="lime")
pvplt.add_point_labels([nasion_n], ["Nz"], font_size=16,
                       text_color="lime", shape=None, always_visible=True)

# Eye positions (for reference)
pvplt.add_mesh(pv.Sphere(radius=3, center=r_eye_n), color="cyan")
pvplt.add_mesh(pv.Sphere(radius=3, center=l_eye_n), color="cyan")

# MediaPipe face oval (orange points) -- rotate into canonical frame
if face_contour is not None:
    contour_n = (R_eye @ (np.asarray(face_contour) - rot_center).T).T + rot_center
    pvplt.add_mesh(
        pv.PolyData(contour_n), color="orange", point_size=14,
        render_points_as_spheres=True,
        label=f"MP face oval ({len(contour_n)} pts)",
    )

pvplt.add_text(
    f"S{SUBJECT_NUMBER} | deletion region: {extended_mask.sum():,} vertices",
    position="upper_left", font_size=12,
)
pvplt.add_legend()
pvplt.show()


Widget(value='<iframe src="http://localhost:40923/index.html?ui=P_0x7fc9d27b2350_0&reconnect=auto" class="pyvi…

## 4. Interactive refinement — click to add or remove deletion zones

- **Left-click**: Add/remove vertices within the brush radius
- **T**: Toggle between **add** mode (mark for deletion) and **remove** mode (undo)
- **+** / **-**: Increase / decrease brush radius
- Close the window when done.

In [5]:
refined_mask = extended_mask.copy()
click_count = [0]
brush_radius = [CIRCLE_RADIUS]  # mutable so key events can change it
mode = ['add']  # 'add' or 'remove'

# Protected zone: vertices near landmarks that cannot be added to mask
protection_radius = 15.0  # mm

def update_title():
    """Update the on-screen status text."""
    pvplt.add_text(
        f'Mode: {mode[0].upper()}  |  Brush: {brush_radius[0]:.0f}mm\n'
        f'T = toggle add/remove  |  +/- = brush size',
        position='lower_right', font_size=14, name='status',
        color='black',
    )

def on_pick(picked_point):
    if picked_point is None:
        return
    point = np.array(picked_point)

    # Find all vertices within brush radius
    dists = np.linalg.norm(verts - point, axis=1)
    circle = dists < brush_radius[0]

    if mode[0] == 'add':
        # Check if inside protected zone
        dist_to_protected = np.linalg.norm(protected_positions - point, axis=1)
        if np.any(dist_to_protected < protection_radius):
            print('  Skipped -- too close to a protected landmark')
            return

        new_verts = circle & ~refined_mask
        n_changed = new_verts.sum()
        if n_changed == 0:
            return
        refined_mask[new_verts] = True

        # Yellow sphere for added regions
        sphere = pv.Sphere(radius=brush_radius[0], center=point)
        pvplt.add_mesh(sphere, color='yellow', opacity=0.2)
    else:
        # Remove mode -- unmark vertices
        removed = circle & refined_mask
        n_changed = removed.sum()
        if n_changed == 0:
            return
        refined_mask[removed] = False

        # Blue sphere for removed regions
        sphere = pv.Sphere(radius=brush_radius[0], center=point)
        pvplt.add_mesh(sphere, color='cyan', opacity=0.2)

    click_count[0] += 1

    # Update mask coloring on mesh (in-place)
    pv_mesh['mask'] = refined_mask.astype(float)

    sign = '+' if mode[0] == 'add' else '-'
    print(f'  Click {click_count[0]} ({mode[0]}): {sign}{n_changed:,} vertices '
          f'(total: {refined_mask.sum():,})')

def toggle_mode():
    mode[0] = 'remove' if mode[0] == 'add' else 'add'
    update_title()
    print(f'  Mode: {mode[0].upper()}')

def increase_radius():
    brush_radius[0] = min(brush_radius[0] + 5, 100)
    update_title()
    print(f'  Brush radius: {brush_radius[0]:.0f}mm')

def decrease_radius():
    brush_radius[0] = max(brush_radius[0] - 5, 5)
    update_title()
    print(f'  Brush radius: {brush_radius[0]:.0f}mm')

# Build plotter (non-notebook for interactive picking)
pvplt = pv.Plotter(notebook=False)

vtk_surface = VTKSurface.from_trimeshsurface(surface_h)
pv_mesh = pv.wrap(vtk_surface.mesh)

# Color mesh by mask: white=keep, red=delete
pv_mesh['mask'] = refined_mask.astype(float)
pvplt.add_mesh(pv_mesh, scalars='mask', cmap=['white', 'red'],
               clim=[0, 1], show_scalar_bar=False, opacity=0.9,
               smooth_shading=True, pickable=True)

# Show protected landmarks
for label, pos in zip(protected_labels, protected_positions):
    pvplt.add_mesh(pv.Sphere(radius=3, center=pos), color='lime')
    pvplt.add_point_labels([pos], [label], font_size=12, point_size=0,
                           text_color='lime', shape=None, always_visible=True)

pvplt.enable_surface_point_picking(
    callback=on_pick,
    left_clicking=True,
    show_point=False,
    tolerance=0.005,
)

# Key bindings
pvplt.add_key_event('t', toggle_mode)
pvplt.add_key_event('plus', increase_radius)
pvplt.add_key_event('minus', decrease_radius)

update_title()
pvplt.show()

# After window closes
extended_mask = refined_mask
print(f'\nDone -- {click_count[0]} clicks, final mask: {extended_mask.sum():,} vertices')


## 5. Anonymize — delete facial vertices

In [6]:
# Delete faces where ANY vertex is in the mask
mesh_copy = surface_h.mesh.copy()
face_verts_in_mask = extended_mask[mesh_copy.faces]
faces_to_remove = face_verts_in_mask.any(axis=1)
mesh_copy.update_faces(~faces_to_remove)
mesh_copy.remove_unreferenced_vertices()

n_removed = surface_h.nvertices - len(mesh_copy.vertices)
print(f'Original:    {surface_h.nvertices:,} verts, {surface_h.nfaces:,} faces')
print(f'Anonymized:  {len(mesh_copy.vertices):,} verts, {len(mesh_copy.faces):,} faces')
print(f'Removed:     {n_removed:,} verts')

# Side-by-side visualization
anon_vtk_mesh = trimesh_to_vtk_polydata(mesh_copy)

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, surface_h, opacity=1.0)
pvplt.add_text(f'S{SUBJECT_NUMBER} Original', position='upper_left', font_size=14)

pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk_mesh), rgb=True)
pvplt.add_text(f'S{SUBJECT_NUMBER} Anonymized (-{n_removed:,} verts)',
               position='upper_left', font_size=14)

pvplt.link_views()
pvplt.show()

Original:    391,330 verts, 741,306 faces
Anonymized:  233,308 verts, 441,792 faces
Removed:     158,022 verts


Widget(value='<iframe src="http://localhost:36161/index.html?ui=P_0x7f2042b29090_1&reconnect=auto" class="pyvi…

## 6. Anonymized head — original color

In [8]:
pvplt = pv.Plotter()
pvplt.add_mesh(pv.wrap(anon_vtk_mesh), rgb=True, smooth_shading=True)
pvplt.add_text(f'S{SUBJECT_NUMBER} Anonymized', position='upper_left', font_size=14)
pvplt.show()

Widget(value='<iframe src="http://localhost:42885/index.html?ui=P_0x7f8fe792ab90_2&reconnect=auto" class="pyvi…